# Week 25: AR for satellites: sky-direction overlays and az-el math

**Track:** Space GIS Architect (Expert)  
**Full primer + quiz:** [https://launchdetect.com/academy/week/25/](https://launchdetect.com/academy/week/25/)  
**Track index:** [https://launchdetect.com/academy/space-gis-architect/](https://launchdetect.com/academy/space-gis-architect/)

---

_Augmented reality for the sky. LaunchDetect's mobile app puts ISS, Starlink, and Hubble into your viewfinder. This week is the math + code that powers it._


## Why this week matters

AR is how a non-technical user experiences orbital geometry. Point your phone, see the ISS — the magic is in the az/el-to-pixel projection. The math is also useful for ground-station antenna pointing.


## Learning objectives

By the end of this lab you will be able to:

- Compute azimuth and elevation from observer to satellite
- Project that direction into a smartphone camera frame
- Use device orientation sensors to align the overlay
- Build a working AR overlay for satellite spotting


## Setup — and why these dependencies

This lab uses: `leafmap[common] xarray rasterio pystac-client boto3`. Run the cell below once; Colab installs into the runtime.


In [ ]:
# Install required packages
!pip install -q leafmap[common] xarray rasterio pystac-client boto3


## The approach (and why)

Use skyfield to compute az/el from observer + TLE. Compare against the phone's reported azimuth/elevation (from DeviceOrientation API in a browser). If the deltas are within FOV, the satellite is on-screen.


In [ ]:
# Week 25: az/el math for AR satellite spotting.
from skyfield.api import EarthSatellite, load, wgs84
import math

tle1 = "1 25544U 98067A   24130.50145833  .00018539  00000-0  33188-3 0  9994"
tle2 = "2 25544  51.6406 348.5395 0006703 117.9568 358.1729 15.50289267449420"
ts = load.timescale()
iss = EarthSatellite(tle1, tle2, "ISS", ts)

# Observer + viewport assumptions
observer = wgs84.latlon(40.7128, -74.0060)   # NYC
phone_az_deg = 90   # phone pointing east
phone_el_deg = 30   # tilted up 30°
fov_h_deg, fov_v_deg = 67, 52

t = ts.now()
alt, az, _ = (iss - observer).at(t).altaz()
sat_el, sat_az = alt.degrees, az.degrees

# Angular delta
d_az = ((sat_az - phone_az_deg) + 180) % 360 - 180
d_el = sat_el - phone_el_deg

in_view = abs(d_az) < fov_h_deg/2 and abs(d_el) < fov_v_deg/2
print(f"ISS: az={sat_az:.1f}°, el={sat_el:.1f}°")
print(f"Delta from phone: Δaz={d_az:.1f}°, Δel={d_el:.1f}°  in_view={in_view}")

# TODO: build a browser-based AR demo that reads device orientation, applies
# magnetic declination correction, and overlays the ISS at the right pixel.


## What just happened — and why it works

The DeviceOrientation API gives alpha (compass heading), beta (front-back tilt), gamma (left-right tilt). The phone's alpha is referenced to magnetic north — you must correct to true north by adding local magnetic declination.


## Common gotchas

- iOS requires explicit DeviceOrientationEvent.requestPermission() — Android does not. Code defensively.
- Magnetic declination changes with location and time. Use a current WMM/IGRF lookup.
- Camera FOV varies by device. iPhone main camera is ~67° horizontal; ultra-wide is ~120°. Detect and adapt.


## Self-check

Verify your solution against these checks before considering the lab complete:

- [ ] Output type matches the expected format (GeoJSON / PNG / table / etc.).
- [ ] No exceptions raised on a clean run.
- [ ] Result is visually plausible — open the map cell, scan the values, sanity-check magnitudes.
- [ ] Quiz on the [week page](https://launchdetect.com/academy/week/25/) — try answering before checking the key.

---

Found a bug or want to contribute an improvement? Open an issue or PR at [github.com/launchdetect/academy-labs](https://github.com/launchdetect/academy-labs).
